📅 **论文年份 (Year):2024 年**  
*Better & Faster Large Language Models via Multi-token Prediction — Gloeckle et al. (Meta)*

# Paper 27: Better & Faster Large Language Models via Multi-token Prediction(论文 27:通过多 token 预测构建更好、更快的大语言模型)
## Meta AI Research (2024)(Meta AI 研究(2024))

### Multi-token Prediction(多 token 预测)

Key insight: Train LMs to predict multiple future tokens simultaneously. Improves sample efficiency and generation quality!

关键洞见:训练语言模型(LM)同时预测多个未来 token。这能提升样本效率和生成质量!

## 📖 论文导读

**🎯 这篇文章想解决什么问题(目的):** 现在的大语言模型(比如 ChatGPT 背后的模型)在训练时都在做同一件事:看一段文字,猜下一个词是什么,猜完一个再猜下一个。这就像学开车时只盯着车头前一米的路面,永远不往远处看。这种"一次只猜一个词"的方式有两个毛病:训练时每段文字只能提供很少的学习信号,数据利用率低;生成文字时也只能一个词一个词地慢慢吐,速度上不去。

**💡 主要贡献:** Meta 的研究者提出了一个简单却有效的改进:训练时让模型一次预测未来的多个词(比如接下来的 4 个词),而不是只预测 1 个。他们用实验证明,这样训练出来的模型不仅没有变差,反而在代码生成等任务上明显更强,而且同样的数据能"榨出"更多的学习价值——用三分之一的数据就能达到原来的效果。

**🔧 方法:** 具体做法是给模型装上多个"预测头":模型的主体部分(负责理解上下文)完全共享,只在最后输出的地方并排接上几个小型输出层,第 1 个头猜下一个词,第 2 个头猜下下个词,以此类推。这就像一个厨师(主干网络)备好菜之后,由几个帮手(预测头)同时装出好几盘。训练时把各个头的误差加在一起一起学习;真正使用时既可以只用第 1 个头(和普通模型一样),也可以让多个头一起出词、再快速验证,一次前向计算生成好几个词。

**🌟 意义:** 这项工作说明"预测更远的未来"能逼着模型学到更深层的规律,而不是只会背下一个词——这相当于免费的正则化,还附带推理加速的红利(配合投机解码可以快约 3 倍)。它启发了后来许多加速生成的技术,DeepSeek-V3 等知名模型也采用了多 token 预测来提升训练效果。对理解现代大模型"如何训练得更省、跑得更快",这是一篇入门必读的论文。

## 🎯 核心结论 (Key Takeaways)

- **论文核心思想:一次多猜几个词。** 传统语言模型每步只预测下一个 token;Meta 的做法是在共享主干上并排接多个小输出头,同时预测未来 4 个(本 notebook 演示 3 个)token,训练开销几乎不变。
- **论文核心发现:预测更远 = 学得更好、更省数据。** 在 7B/13B 规模上,多 token 训练让代码生成等任务明显更强(困惑度约降到标准方法的 0.7 倍),且约用 1/3 的数据就能达到标准训练的效果——相当于免费的正则化。
- **附赠推理加速:** 多个头一次前向就能吐出多个候选 token,配合投机解码(自己起草、自己并行验证)可把生成速度提到约 **3 倍**;DeepSeek-V3 等后来的模型也采用了这一思路。
- **本 notebook 演示了什么:** 用纯 NumPy 实现的 `MultiTokenRNN`(3 个预测头)对比 `SingleTokenRNN`——同一次前向计算,单 token 模型每个位置只给 1 个预测,多 token 模型给出 t+1、t+2、t+3 共 **3 个预测**,即每个样本提供约 3 倍的训练信号,这正是论文“样本效率”优势的来源。
- **注意实验数字的解读:** 本 notebook 是简化教学实现,训练函数只统计损失、**没有实现反向传播**,所以两条学习曲线都停在随机水平(loss ≈ ln 50 ≈ 3.91,准确率约 2% 的随机基线)。它演示的是“训练信号从哪来、结构长什么样”,真实的收敛对比请以论文数字为准。
- **带走一句话:** 让模型“看得更远”几乎没有额外代价,却同时换来更好的表示、更高的数据利用率和最高 N 倍的生成加速——既然可以多预测几个 token,为什么只预测一个?

## 🤯 反常识的发现 (Counterintuitive Findings)

- **常识认为:让模型同时猜 4 个词,肯定比只猜 1 个更难、更容易错,会拖累学习。** 但论文发现恰恰相反:这份"更难的作业"逼着模型去理解更长远的结构(比如代码里"写了 `if` 之后迟早要写对应的逻辑块"),反而学得更好——13B 模型在代码任务上比单 token 训练多解出约 12% 的题。本 notebook 里也能看到这一点的雏形:`MultiTokenRNN` 的 3 个头让同一次前向传播产出 3 倍的训练信号,"样本效率对比"实验展示的正是这份免费加量的监督。
- **常识认为:加了 3 个额外预测头,推理时肯定更慢、更贵。** 但这些头训练完可以直接扔掉,只留主干和第一个头,推理成本和普通模型一模一样;或者反过来把它们用于自我投机解码——自己起草、自己并行验证,生成速度提到约 **3 倍**。"白拿好处、不付账单",这正是 DeepSeek-V3 后来沿用这一思路的原因。
- **常识认为:好方法应该对大小模型一视同仁。** 但论文发现多 token 预测在小模型上几乎没有收益甚至略有伤害,模型越大收益越明显——规模改变了任务性质:小模型连"猜下一个词"都自顾不暇,大模型才有富余容量去消化"看得更远"带来的额外信号。
- **常识认为:模型多学到的东西应该体现在损失数字上。** 但注意本 notebook 的训练函数只统计损失、没有实现反向传播,两条学习曲线都停在随机水平(loss ≈ 3.91)——多 token 的真正优势不在这条曲线上,而在"每个位置多出 2 个预测目标"的结构本身;真实的收敛差距要看论文里 7B/13B 规模的实验。

#### 💻 代码解读

**做什么:** 导入本笔记本需要的工具库,并固定随机种子,保证实验结果可以复现。

**怎么做:**
- 导入 `numpy`(数值计算库,负责矩阵运算)和 `matplotlib.pyplot`(画图库,负责后面画曲线和热力图)。
- 调用 `np.random.seed(42)` 固定随机数种子——就像掷骰子前先"锁定"骰子的顺序,每次运行都会得到一模一样的随机数,方便对比实验。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## Standard Single-Token Prediction(标准单 token 预测)

Traditional language modeling:
```
Input:  [w1, w2, w3, w4]
Predict: w5
```

传统的语言建模:给定输入 `[w1, w2, w3, w4]`,预测下一个 token `w5`。

#### 💻 代码解读

**做什么:** 实现传统的"单 token 预测"RNN 模型(`SingleTokenRNN`),它每次只猜下一个词,作为后面对比的基准。

**怎么做:**
- 先定义 `softmax` 函数:把模型输出的原始分数(logits)变成加起来等于 1 的概率分布,就像把"喜好程度"换算成"投票比例"。
- `SingleTokenRNN` 的 `__init__` 里初始化了几组权重:词嵌入表 `W_embed`(把词编号变成向量)、RNN 循环权重 `W_xh`/`W_hh`(负责记住上下文),以及一个输出头 `W_out`/`b_out`(只负责预测下一个词)。
- `forward` 方法逐个读入 token:先查嵌入表得到向量 `x`,再用 `tanh` 更新隐藏状态 `h`(相当于随时更新的"记忆"),最后用输出头算出下一个词的概率。
- 末尾用长度为 4 的测试序列 `[1, 2, 3, 4]` 跑一遍,打印结果确认:每个位置只预测 1 个未来 token。

In [ ]:
def softmax(x):
    # 数值稳定技巧:先减去每行最大值再取exp,防止exp溢出;keepdims=True保持维度以便广播
    exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

class SingleTokenRNN:
    """Standard RNN with single-token prediction"""
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        
        # Embeddings
        # 嵌入表:每个token对应一行向量;乘0.01做小随机初始化,避免初始激活过大
        self.W_embed = np.random.randn(vocab_size, embedding_dim) * 0.01
        
        # RNN weights
        self.W_xh = np.random.randn(hidden_dim, embedding_dim) * 0.01
        self.W_hh = np.random.randn(hidden_dim, hidden_dim) * 0.01
        self.b_h = np.zeros((hidden_dim, 1))
        
        # Output projection (predict next token)
        # 单token基线:只有一个输出头,把隐藏状态映射为下一个token的logits
        self.W_out = np.random.randn(vocab_size, hidden_dim) * 0.01
        self.b_out = np.zeros((vocab_size, 1))
    
    def forward(self, input_seq):
        """
        Forward pass
        input_seq: list of token indices
        Returns: predictions for next token at each position
        """
        h = np.zeros((self.hidden_dim, 1))
        predictions = []
        hidden_states = []
        
        for token_idx in input_seq:
            # Embed
            # 按行索引取出该token的嵌入,并reshape成列向量,形状:(embedding_dim, 1)
            x = self.W_embed[token_idx].reshape(-1, 1)
            
            # RNN step
            # 经典RNN递推:新隐藏状态=tanh(输入变换+上一步隐藏状态变换+偏置)
            h = np.tanh(np.dot(self.W_xh, x) + np.dot(self.W_hh, h) + self.b_h)
            
            # Predict next token
            # 形状:(vocab_size, hidden_dim) @ (hidden_dim, 1) -> (vocab_size, 1)
            logits = np.dot(self.W_out, h) + self.b_out
            # 转置成行向量后做softmax,得到词表上的概率分布
            probs = softmax(logits.T)
            
            predictions.append(probs.flatten())
            hidden_states.append(h.copy())
        
        return predictions, hidden_states

# Test
vocab_size = 50
single_model = SingleTokenRNN(vocab_size, embedding_dim=32, hidden_dim=64)
test_seq = [1, 2, 3, 4]
preds, _ = single_model.forward(test_seq)
print(f"Input sequence length: {len(test_seq)}")
print(f"Predictions shape: {len(preds)} x {len(preds[0])}")
print(f"Predicts: 1 token ahead at each position")

## Multi-Token Prediction(多 token 预测)

Predict multiple future tokens:
```
Input:  [w1, w2, w3, w4]
Predict: w5, w6, w7  (3 tokens ahead!)
```

预测多个未来 token:给定输入 `[w1, w2, w3, w4]`,同时预测 `w5, w6, w7`(向前 3 个 token!)。

#### 💻 代码解读

**做什么:** 实现本论文的核心思想——"多 token 预测"模型(`MultiTokenRNN`),让模型在每个位置同时猜未来 3 个词,而不是只猜 1 个。

**怎么做:**
- 网络主干(嵌入表 `W_embed` 和 RNN 权重 `W_xh`/`W_hh`)与单 token 版完全一样,是共享的"大脑"。
- 关键区别在输出层:用循环建了 `num_future_tokens=3` 个独立输出头,存进 `self.output_heads` 列表——就像一个大脑接了 3 张嘴,分别负责说出 t+1、t+2、t+3 的预测。
- `forward` 方法里,每读入一个 token、更新一次隐藏状态 `h` 后,依次让 3 个输出头各算一个概率分布,打包成 `position_preds` 存入 `multi_predictions`。
- 末尾用同样的测试序列验证:每个位置输出 3 个未来 token 的预测,每个预测都是 50 维(词表大小)的概率向量。

In [ ]:
class MultiTokenRNN:
    """RNN with multi-token prediction"""
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_future_tokens=3):
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        self.num_future_tokens = num_future_tokens
        
        # Shared embeddings and RNN
        self.W_embed = np.random.randn(vocab_size, embedding_dim) * 0.01
        self.W_xh = np.random.randn(hidden_dim, embedding_dim) * 0.01
        self.W_hh = np.random.randn(hidden_dim, hidden_dim) * 0.01
        self.b_h = np.zeros((hidden_dim, 1))
        
        # Multiple output heads (one per future position)
        # 论文核心思想:共享同一个主干(嵌入+RNN),只为t+1、t+2、...、t+N各配一个独立输出头
        self.output_heads = []
        for i in range(num_future_tokens):
            W_out = np.random.randn(vocab_size, hidden_dim) * 0.01
            b_out = np.zeros((vocab_size, 1))
            # 每个头是一组(权重,偏置)元组,参数互不共享
            self.output_heads.append((W_out, b_out))
    
    def forward(self, input_seq):
        """
        Forward pass
        Returns: predictions for next N tokens at each position
        """
        h = np.zeros((self.hidden_dim, 1))
        multi_predictions = []  # List of (pred_t+1, pred_t+2, ..., pred_t+N)
        hidden_states = []
        
        for token_idx in input_seq:
            # Embed
            x = self.W_embed[token_idx].reshape(-1, 1)
            
            # RNN step
            h = np.tanh(np.dot(self.W_xh, x) + np.dot(self.W_hh, h) + self.b_h)
            
            # Predict next N tokens using separate heads
            # 同一个隐藏状态h被N个头复用:h必须同时编码近期和更远未来的信息
            position_preds = []
            # 元组解包:依次取出第i个头的权重和偏置,分别预测t+1+i位置的token
            for W_out, b_out in self.output_heads:
                logits = np.dot(W_out, h) + b_out
                probs = softmax(logits.T)
                position_preds.append(probs.flatten())
            
            multi_predictions.append(position_preds)
            hidden_states.append(h.copy())
        
        return multi_predictions, hidden_states

# Test
multi_model = MultiTokenRNN(vocab_size, embedding_dim=32, hidden_dim=64, num_future_tokens=3)
multi_preds, _ = multi_model.forward(test_seq)
print(f"Input sequence length: {len(test_seq)}")
print(f"Multi-predictions: {len(multi_preds)} positions")
print(f"At each position: {len(multi_preds[0])} future tokens")
print(f"Each prediction shape: {multi_preds[0][0].shape}")
print(f"\nPredicts: {len(multi_preds[0])} tokens ahead at each position!")

## Synthetic Text Data(合成文本数据)

#### 💻 代码解读

**做什么:** 用 `generate_synthetic_sequences` 函数造一批有规律的"人工文本"数据,供两个模型训练和测试。

**怎么做:**
- 每条序列都是等差数列:随机选一个起点 `start` 和步长 `step`(1 或 2),按 `(start + i * step) % vocab_size` 生成 20 个数——就像 "3, 5, 7, 9…" 这样规律明显、模型学得会的"句子"。
- 取模 `% vocab_size` 保证数字不超出词表范围(0~49),超出就绕回来。
- 生成 1000 条训练序列 `train_sequences` 和 200 条测试序列 `test_sequences`,并打印一条样例确认数据长什么样。

In [ ]:
def generate_synthetic_sequences(vocab_size=50, num_sequences=1000, seq_length=20):
    """
    Generate synthetic sequences with patterns
    Pattern: arithmetic progressions (e.g., 1, 2, 3, 4, ...)
    """
    sequences = []
    
    for _ in range(num_sequences):
        # Random starting point and step
        # 规律性强的合成数据:等差数列让"未来多个token"完全可预测,便于对比两种训练方式
        start = np.random.randint(0, vocab_size // 2)
        step = np.random.randint(1, 3)
        
        # Generate arithmetic sequence
        # 列表推导式生成等差数列,取模%让token索引在词表范围内回绕
        seq = [(start + i * step) % vocab_size for i in range(seq_length)]
        sequences.append(seq)
    
    return sequences

# Generate data
train_sequences = generate_synthetic_sequences(vocab_size, num_sequences=1000, seq_length=20)
test_sequences = generate_synthetic_sequences(vocab_size, num_sequences=200, seq_length=20)

print(f"Training sequences: {len(train_sequences)}")
print(f"Example sequence: {train_sequences[0][:10]}...")
print(f"Pattern: arithmetic progression")

## Training: Single-Token Prediction(训练:单 token 预测)

#### 💻 代码解读

**做什么:** 定义 `train_single_token` 函数训练单 token 模型:标准的"看前文、猜下一个词"训练方式,并记录损失变化。

**怎么做:**
- 对每条序列的每个位置 `i`,取前面的 `input_tokens = seq[:i+1]` 作为输入,把紧接着的 `seq[i+1]` 作为标准答案 `target_token`。
- 前向计算后取最后一个位置的预测概率 `pred_probs`,用交叉熵 `-np.log(pred_probs[target_token])` 算损失——猜对的概率越低,罚分越高(注:这里做了简化,只统计损失,没有真正反向传播更新权重)。
- 每个 epoch 把损失取平均存入 `losses` 列表,每 10 轮打印一次进度。
- 最后用前 100 条训练序列跑 30 轮,得到 `single_losses` 供后面画学习曲线。

In [ ]:
def train_single_token(model, sequences, epochs=50, lr=0.01):
    """
    Train with standard next-token prediction
    """
    losses = []
    
    for epoch in range(epochs):
        epoch_loss = 0
        
        for seq in sequences:
            # Predict next token at each position
            for i in range(len(seq) - 1):
                # 切片seq[:i+1]取前缀作为输入,紧随其后的seq[i+1]是要预测的目标
                input_tokens = seq[:i+1]
                target_token = seq[i+1]
                
                # Forward
                predictions, _ = model.forward(input_tokens)
                # 只取序列最后一个位置的预测,它对应"下一个token"
                pred_probs = predictions[-1]  # Last position prediction
                
                # Loss
                # 交叉熵损失:-log(正确token的概率);加1e-8防止log(0)
                loss = -np.log(pred_probs[target_token] + 1e-8)
                epoch_loss += loss
                
                # Backward (simplified - just track loss)
        
        avg_loss = epoch_loss / (len(sequences) * (len(seq) - 1))
        losses.append(avg_loss)
        
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")
    
    return losses

# Train single-token model
print("Training Single-Token Model...\n")
single_losses = train_single_token(single_model, train_sequences[:100], epochs=30)
print(f"\nFinal loss: {single_losses[-1]:.4f}")

## Training: Multi-Token Prediction(训练:多 token 预测)

#### 💻 代码解读

**做什么:** 定义 `train_multi_token` 函数训练多 token 模型:一次同时对未来 3 个词计算损失,给模型更丰富的学习信号。

**怎么做:**
- 与单 token 训练类似,但每个位置的标准答案是接下来的 3 个词 `target_tokens = seq[i+1:i+1+num_future_tokens]`,而不是 1 个。
- 前向计算后取最后位置的 3 份预测 `position_preds`,用 `zip` 把每份预测和对应答案配对,分别算交叉熵损失再累加——相当于一道题同时批改 3 个空,反馈信息量是原来的 3 倍。
- 用 `num_predictions` 计数,把总损失除以预测次数得到平均损失,保证和单 token 的损失在同一尺度上可比。
- 最后用同样的 100 条序列训练 30 轮,得到 `multi_losses`,并打印最终损失。

In [ ]:
def train_multi_token(model, sequences, epochs=50, lr=0.01):
    """
    Train with multi-token prediction
    Loss = sum of losses for all future positions
    """
    losses = []
    
    for epoch in range(epochs):
        epoch_loss = 0
        num_predictions = 0
        
        for seq in sequences:
            # Predict multiple tokens at each position
            # 循环上界减去num_future_tokens,保证后面切片能取满N个目标token
            for i in range(len(seq) - model.num_future_tokens):
                input_tokens = seq[:i+1]
                # 切片取紧随其后的N个token作为多目标,如N=3时目标是t+1,t+2,t+3
                target_tokens = seq[i+1:i+1+model.num_future_tokens]
                
                # Forward
                multi_preds, _ = model.forward(input_tokens)
                position_preds = multi_preds[-1]  # Last position predictions
                
                # Loss for each future position
                # zip把N个头的预测和N个目标一一配对;总损失=各未来位置交叉熵之和(论文式的多任务损失)
                for j, (pred_probs, target) in enumerate(zip(position_preds, target_tokens)):
                    loss = -np.log(pred_probs[target] + 1e-8)
                    epoch_loss += loss
                    num_predictions += 1
        
        # 按预测次数归一化,这样才能和单token模型的平均损失公平比较;条件表达式防除零
        avg_loss = epoch_loss / num_predictions if num_predictions > 0 else 0
        losses.append(avg_loss)
        
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")
    
    return losses

# Train multi-token model
print("\nTraining Multi-Token Model (3 tokens ahead)...\n")
multi_losses = train_multi_token(multi_model, train_sequences[:100], epochs=30)
print(f"\nFinal loss: {multi_losses[-1]:.4f}")

## Compare Learning Curves(学习曲线对比)

#### 💻 代码解读

**做什么:** 把前面两种训练方式的损失曲线画在同一张图上,直观对比单 token 和多 token 预测的学习过程。

**怎么做:**
- 用 `plt.plot` 分别画出 `single_losses`(圆点标记)和 `multi_losses`(方块标记)随 epoch 变化的曲线。
- 加上坐标轴标签、标题、图例和网格,让图表一目了然。
- 最后打印两个模型的最终损失数值,结论是:多 token 预测在每一步提供了更丰富的训练信号(一次学 3 个答案 vs 只学 1 个)。

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(single_losses, label='Single-Token Prediction', linewidth=2, marker='o', markersize=4)
plt.plot(multi_losses, label='Multi-Token Prediction (3 ahead)', linewidth=2, marker='s', markersize=4)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Average Loss', fontsize=12)
plt.title('Learning Curves: Single vs Multi-Token Prediction', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nSingle-token final loss: {single_losses[-1]:.4f}")
print(f"Multi-token final loss: {multi_losses[-1]:.4f}")
print(f"\nMulti-token prediction provides richer training signal!")

## Evaluation: Prediction Accuracy(评估:预测准确率)

#### 💻 代码解读

**做什么:** 定义两个评估函数,在测试集上量化对比两个模型的预测准确率,看看多 token 模型在不同"预测距离"下表现如何。

**怎么做:**
- `evaluate_single_token`:对每个位置用 `np.argmax` 取模型认为概率最大的词,和真实的下一个词对比,统计猜对的比例。
- `evaluate_multi_token`:多了一个 `position` 参数(0/1/2 分别代表 t+1/t+2/t+3),取 `multi_preds[-1][position]` 这个输出头的预测,和对应距离的真实词对比——相当于分别给 3 张"嘴"打分。
- 在 50 条测试序列上算出 4 个准确率:单 token 模型的 t+1,以及多 token 模型的 t+1、t+2、t+3。
- 打印一张对比表,预期规律是:预测得越远,难度越大,准确率越低。

In [ ]:
def evaluate_single_token(model, sequences):
    """Evaluate next-token prediction accuracy"""
    correct = 0
    total = 0
    
    for seq in sequences:
        for i in range(len(seq) - 1):
            input_tokens = seq[:i+1]
            target = seq[i+1]
            
            predictions, _ = model.forward(input_tokens)
            # 贪心解码:argmax取概率最大的token作为预测结果
            pred_token = np.argmax(predictions[-1])
            
            if pred_token == target:
                correct += 1
            total += 1
    
    return correct / total if total > 0 else 0

def evaluate_multi_token(model, sequences, position=0):
    """Evaluate multi-token prediction accuracy at specific future position"""
    correct = 0
    total = 0
    
    for seq in sequences:
        for i in range(len(seq) - model.num_future_tokens):
            input_tokens = seq[:i+1]
            # position=0对应t+1,position=1对应t+2,以此类推
            target = seq[i+1+position]
            
            multi_preds, _ = model.forward(input_tokens)
            # 双重索引:[-1]取最后一个输入位置,[position]取该位置第position个未来头的输出
            pred_probs = multi_preds[-1][position]  # Prediction for position ahead
            pred_token = np.argmax(pred_probs)
            
            if pred_token == target:
                correct += 1
            total += 1
    
    return correct / total if total > 0 else 0

# Evaluate both models
single_acc = evaluate_single_token(single_model, test_sequences[:50])
multi_acc_t1 = evaluate_multi_token(multi_model, test_sequences[:50], position=0)
multi_acc_t2 = evaluate_multi_token(multi_model, test_sequences[:50], position=1)
multi_acc_t3 = evaluate_multi_token(multi_model, test_sequences[:50], position=2)

print("\nEvaluation Results:")
print(f"{'='*60}")
print(f"Single-Token Model:")
print(f"  Next token (t+1): {single_acc:.2%}")
print(f"\nMulti-Token Model:")
print(f"  Next token (t+1): {multi_acc_t1:.2%}")
print(f"  2 tokens ahead (t+2): {multi_acc_t2:.2%}")
print(f"  3 tokens ahead (t+3): {multi_acc_t3:.2%}")
print(f"{'='*60}")

## Visualize Multi-Token Predictions(多 token 预测可视化)

#### 💻 代码解读

**做什么:** 用一条具体的测试序列,可视化多 token 模型在每个位置对 t+1、t+2、t+3 的预测是对是错,并统计不同预测距离的平均准确率。

**怎么做:**
- 取测试序列前 15 个 token,逐位置调用 `multi_model.forward`,把每个输出头的 `argmax` 预测和真实答案对比,结果(对=1,错=0)填进 `accuracies` 矩阵。
- 左图用 `imshow` 画热力图:横轴是输入位置,纵轴是 t+1/t+2/t+3,绿色代表猜对、红色代表猜错,哪里出错一眼可见。
- 右图用 `ax2.bar` 画柱状图:对每个预测距离取平均准确率 `avg_accs`,并在柱子上标注百分比数字。
- 结论:预测越远的 token 越难猜准(t+1 最准,t+3 最差),符合直觉。

In [ ]:
# Generate prediction accuracy heatmap
test_seq = test_sequences[0][:15]
# 准确率矩阵,形状:(输入位置数, 3个未来位置),元素为0/1表示预测对错
accuracies = np.zeros((len(test_seq) - 3, 3))

for i in range(len(test_seq) - 3):
    input_tokens = test_seq[:i+1]
    targets = test_seq[i+1:i+4]
    
    multi_preds, _ = multi_model.forward(input_tokens)
    position_preds = multi_preds[-1]
    
    for j in range(3):
        pred_token = np.argmax(position_preds[j])
        # 条件表达式把布尔判断转成0/1填入矩阵第i行第j列
        accuracies[i, j] = 1.0 if pred_token == targets[j] else 0.0

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap
# 转置.T让横轴是输入位置、纵轴是未来位置,便于阅读热力图
im = ax1.imshow(accuracies.T, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
ax1.set_xlabel('Input Position', fontsize=12)
ax1.set_ylabel('Future Position', fontsize=12)
ax1.set_title('Multi-Token Prediction Accuracy', fontsize=13, fontweight='bold')
ax1.set_yticks([0, 1, 2])
ax1.set_yticklabels(['t+1', 't+2', 't+3'])
plt.colorbar(im, ax=ax1, label='Accuracy (1=Correct, 0=Wrong)')

# Average accuracy by distance
# axis=0沿输入位置求均值,得到每个预测距离(t+1/t+2/t+3)的平均准确率
avg_accs = np.mean(accuracies, axis=0)
positions = ['t+1', 't+2', 't+3']
bars = ax2.bar(positions, avg_accs, color=['green', 'orange', 'red'], edgecolor='black', linewidth=2)
ax2.set_ylabel('Average Accuracy', fontsize=12)
ax2.set_title('Accuracy vs Prediction Distance', fontsize=13, fontweight='bold')
ax2.set_ylim([0, 1])
ax2.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar, acc in zip(bars, avg_accs):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'{acc:.1%}', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print("\nFurther predictions are harder (as expected)")

## Sample Efficiency Comparison(样本效率对比)

#### 💻 代码解读

**做什么:** 做"样本效率"实验:在不同大小的训练集(10~200 条序列)上分别训练两种模型,看谁用更少的数据学得更好。

**怎么做:**
- 遍历 `dataset_sizes = [10, 25, 50, 100, 200]`,每个规模下都新建一个 `SingleTokenRNN` 和一个 `MultiTokenRNN`(从零开始,保证公平),各训练 20 轮。
- 把每次训练的最终损失分别存入 `single_final_losses` 和 `multi_final_losses`。
- 用 `plt.plot` 画出"训练数据量 vs 最终损失"的对比曲线,横轴用对数刻度(`plt.xscale('log')`),让小数据量区域也看得清。
- 结论:多 token 预测更"省数据"——因为每个训练样本能同时提供 3 份监督信号,小数据集下学习效率更高。

In [ ]:
# Train on varying dataset sizes
# 样本效率实验:多token目标让每个前缀产生N倍监督信号,预期小数据下优势更明显
dataset_sizes = [10, 25, 50, 100, 200]
single_final_losses = []
multi_final_losses = []

print("Testing sample efficiency...\n")

for size in dataset_sizes:
    print(f"Training on {size} sequences...")
    
    # Single-token
    # 每种数据量都重新初始化模型,保证两种方法从同样的起点公平对比
    single_temp = SingleTokenRNN(vocab_size, embedding_dim=32, hidden_dim=64)
    single_loss = train_single_token(single_temp, train_sequences[:size], epochs=20, lr=0.01)
    single_final_losses.append(single_loss[-1])
    
    # Multi-token
    multi_temp = MultiTokenRNN(vocab_size, embedding_dim=32, hidden_dim=64, num_future_tokens=3)
    multi_loss = train_multi_token(multi_temp, train_sequences[:size], epochs=20, lr=0.01)
    multi_final_losses.append(multi_loss[-1])

# Plot
plt.figure(figsize=(12, 6))
plt.plot(dataset_sizes, single_final_losses, 'o-', linewidth=2, markersize=10, 
        label='Single-Token', color='blue')
plt.plot(dataset_sizes, multi_final_losses, 's-', linewidth=2, markersize=10, 
        label='Multi-Token (3 ahead)', color='red')
plt.xlabel('Number of Training Sequences', fontsize=12)
plt.ylabel('Final Loss', fontsize=12)
plt.title('Sample Efficiency: Single vs Multi-Token', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
# 横轴取对数刻度,让10到200的数据量分布更均匀易读
plt.xscale('log')
plt.tight_layout()
plt.show()

print("\nMulti-token prediction is more sample efficient (learns faster with less data)!")

## Key Takeaways(要点总结)

### Multi-Token Prediction:(多 token 预测:)

**Standard LM**:
```
Given: w1, w2, w3
Predict: w4
Loss: -log P(w4 | w1, w2, w3)
```

**标准语言模型(Standard LM)**:给定 w1、w2、w3,预测 w4,损失为下一个 token 的负对数似然。

**Multi-Token LM**:
```
Given: w1, w2, w3
Predict: w4, w5, w6  (multiple tokens!)
Loss: -log P(w4|w1:3) - log P(w5|w1:3) - log P(w6|w1:3)
```

**多 token 语言模型(Multi-Token LM)**:给定 w1、w2、w3,同时预测 w4、w5、w6(多个 token!),损失为各未来位置负对数似然之和。

### Architecture:(架构:)

**Shared Backbone**:
- Embeddings
- RNN/Transformer layers

**Multiple Output Heads**:
- Head 1: Predicts t+1
- Head 2: Predicts t+2
- Head 3: Predicts t+3
- ...

Each head is a separate linear layer (small overhead!)

**共享主干(Shared Backbone)**:嵌入(embeddings)、RNN/Transformer 层。

**多个输出头(Multiple Output Heads)**:头 1 预测 t+1,头 2 预测 t+2,头 3 预测 t+3,依此类推。

每个头都是一个独立的线性层(开销很小!)

### Benefits:(优点:)

1. **Sample Efficiency** ✅
   - Each example provides N training signals (not just 1)
   - Learns N times faster (approximately)

2. **Better Representations** ✅
   - Forced to encode longer-term dependencies
   - Can't just memorize next token

3. **Faster Inference** ✅
   - Can generate multiple tokens in one forward pass
   - Speculative decoding: verify predictions in parallel

4. **Better Generalization** ✅
   - More training signal → better features
   - Regularization effect

1. **样本效率(Sample Efficiency)** ✅:每个样本提供 N 个训练信号(而不只是 1 个),学习速度大约快 N 倍。
2. **更好的表示(Better Representations)** ✅:模型被迫编码更长程的依赖关系,不能只靠记忆下一个 token。
3. **更快的推理(Faster Inference)** ✅:一次前向传播即可生成多个 token;投机解码(speculative decoding)可并行验证预测。
4. **更好的泛化(Better Generalization)** ✅:更多训练信号 → 更好的特征;具有正则化效果。

### Training:(训练:)

**Loss Function**:
$$
\mathcal{L} = \sum_{i=1}^{N} \lambda_i \cdot \mathcal{L}_{\text{next-token}}(t+i)
$$

Where:
- $N$ = number of future tokens
- $\lambda_i$ = weight for position $i$ (can downweight distant future)

**Typical settings**:
- $N = 3$ or $N = 4$ tokens ahead
- Equal weights: $\lambda_i = 1/N$
- Or decay: $\lambda_i = \gamma^{i-1}$ where $\gamma < 1$

**损失函数**如上式所示。其中:$N$ 为未来 token 的数量;$\lambda_i$ 为位置 $i$ 的权重(可以降低较远未来位置的权重)。

**典型设置**:向前预测 $N = 3$ 或 $N = 4$ 个 token;等权重:$\lambda_i = 1/N$;或按衰减方式:$\lambda_i = \gamma^{i-1}$,其中 $\gamma < 1$。

### Results from Paper (Meta AI):(论文结果(Meta AI):)

**7B model**:
- Standard: X perplexity
- Multi-token (4 ahead): 0.7X perplexity (better!)

**Sample efficiency**:
- Multi-token with 1/3 data = Standard with full data

**Inference speed**:
- 3x faster generation (using speculative decoding)

**7B 模型**:标准方法的困惑度(perplexity)为 X;多 token(向前 4 个)为 0.7X(更好!)。

**样本效率**:多 token 方法用 1/3 的数据 = 标准方法用全部数据。

**推理速度**:生成速度快 3 倍(使用投机解码)。

### Inference Strategies:(推理策略:)

**1. Standard (still valid)**:
```
Use only head 1 (t+1 predictions)
Same as normal autoregressive generation
```

**2. Speculative Decoding**:
```
Generate w4, w5, w6 from heads
Verify each prediction
Keep valid prefix, regenerate rest
→ Up to Nx speedup!
```

**3. Beam Search Enhancement**:
```
Consider multiple future paths simultaneously
Better long-range planning
```

**1. 标准方式(仍然有效)**:只使用头 1(t+1 预测),与普通自回归生成完全相同。

**2. 投机解码(Speculative Decoding)**:用各个头生成 w4、w5、w6,逐一验证每个预测,保留有效前缀、重新生成其余部分 → 最高可达 N 倍加速!

**3. 束搜索增强(Beam Search Enhancement)**:同时考虑多条未来路径,实现更好的长程规划。

### Comparison with Other Techniques:(与其他技术的对比:)

| Technique | Sample Efficiency | Inference Speed | Complexity |
|-----------|------------------|-----------------|------------|
| Standard LM | 1x | 1x | Low |
| Data Augmentation | 1.2x | 1x | Low |
| **Multi-Token** | **2-3x** | **1-3x** | **Low** |
| Distillation | 1.5x | 1.5x | High |

| 技术 | 样本效率 | 推理速度 | 复杂度 |
|-----------|------------------|-----------------|------------|
| 标准语言模型 | 1x | 1x | 低 |
| 数据增强 | 1.2x | 1x | 低 |
| **多 token 预测** | **2-3x** | **1-3x** | **低** |
| 蒸馏(Distillation) | 1.5x | 1.5x | 高 |

### Implementation Tips:(实现技巧:)

1. **Start simple**: N=2 or N=3 tokens
2. **Shared trunk**: Only output heads are separate
3. **Equal weighting**: Unless you have reason to prefer near/far future
4. **Monitor each head**: Track accuracy for each position
5. **Use for speedup**: Speculative decoding in inference

1. **从简单开始**:N=2 或 N=3 个 token
2. **共享主干**:只有输出头是独立的
3. **等权重**:除非你有理由偏向近期/远期未来
4. **监控每个头**:跟踪每个位置的准确率
5. **用于加速**:推理时使用投机解码

### When to Use:(适用场景:)

✅ **Good for**:
- Limited training data
- Want faster inference
- Long sequences (benefits from long-range signal)
- Structured outputs (code, formulas)

❌ **Not ideal for**:
- Very short sequences
- Highly random outputs
- Memory constrained (extra heads add parameters)

✅ **适合**:训练数据有限;需要更快的推理;长序列(受益于长程信号);结构化输出(代码、公式)。

❌ **不太适合**:非常短的序列;高度随机的输出;内存受限的场景(额外的头会增加参数量)。

### Modern Extensions:(现代扩展:)

1. **Adaptive N**: Use different N for different layers
2. **Hierarchical**: Predict next word, next phrase, next sentence
3. **Discrete diffusion**: Multi-step generation
4. **Continuous-time**: Predict at arbitrary future times

1. **自适应 N(Adaptive N)**:不同层使用不同的 N
2. **分层式(Hierarchical)**:预测下一个词、下一个短语、下一个句子
3. **离散扩散(Discrete diffusion)**:多步生成
4. **连续时间(Continuous-time)**:在任意未来时刻进行预测

### Key Insight:(关键洞见:)

**More prediction = More learning signal = Better models**

Multi-token prediction is essentially **free regularization** with **bonus speedup**. Almost no downside!

**"Why predict one token when you can predict many?"** - Meta AI Team

**更多预测 = 更多学习信号 = 更好的模型**

多 token 预测本质上是**免费的正则化**,还附带**额外的加速**。几乎没有缺点!

**“既然可以预测多个 token,为什么只预测一个?”** —— Meta AI 团队